In [5]:
import csv
import json
import re
import sys
import time
from pathlib import Path
from typing import Any, Dict, List
from urllib.parse import urlencode

import requests
import pandas as pd

In [6]:
BASE_SITE = "https://www.rottentomatoes.com"
PAGE_URL_TEMPLATE = BASE_SITE + "/m/{movie_name}/reviews/all-audience"
API_URL_TEMPLATE = BASE_SITE + "/napi/rtcf/v1/movies/{movie_id}/reviews"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/146.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": BASE_SITE + "/",
}

In [7]:
def fetch_movie_page(session: requests.Session, movie_name: str) -> str:
    """Fetch the audience reviews page HTML for the given movie."""
    page_url = PAGE_URL_TEMPLATE.format(movie_name=movie_name)
    response = session.get(page_url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return response.text


def extract_movie_id(html: str) -> str:
    """Extract the movie UUID from the page HTML."""
    patterns = [
        r'"emsId":"([a-f0-9\-]{36})"',
        r'"mediaId":"([a-f0-9\-]{36})"',
        r'"targetId":"([a-f0-9\-]{36})"',
    ]
    for pattern in patterns:
        match = re.search(pattern, html)
        if match:
            return match.group(1)
    raise RuntimeError("Could not find the Rotten Tomatoes movie ID in the page HTML.")


def fetch_reviews_page(
    session: requests.Session,
    movie_id: str,
    after_cursor: str = "",
    page_count: int = 20,
) -> Dict[str, Any]:
    """Fetch one page of audience reviews from the RT API."""
    params = {
        "after": after_cursor,
        "before": "",
        "pageCount": page_count,
        "topOnly": "false",
        "type": "audience",
        "verified": "false",
    }
    url = API_URL_TEMPLATE.format(movie_id=movie_id) + "?" + urlencode(params)
    response = session.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return response.json()


def normalize_review(review: Dict[str, Any], movie_name: str) -> Dict[str, Any]:
    """Flatten a single review JSON object into a clean dictionary."""
    user_info = review.get("user") or {}
    reaction_summary = review.get("reactionSummary") or {}
    audience_score = review.get("audienceScore") or {}

    return {
        "movie_name": movie_name,
        "review_id": review.get("ratingId"),
        "review_text": review.get("review", "").strip(),
        "rating": review.get("rating"),
        "display_name": review.get("displayName"),
        "initials": review.get("initials"),
        "is_verified": review.get("isVerified"),
        "is_super_reviewer": review.get("isSuperReviewer"),
        "has_spoilers": review.get("hasSpoilers"),
        "has_profanity": review.get("hasProfanity"),
        "create_date": review.get("createDate"),
        "audience_score": audience_score.get("score"),
        "user_id": user_info.get("encryptedUserId"),
        "helpful_count": reaction_summary.get("countHelpful"),
        "not_helpful_count": reaction_summary.get("countNotHelpful"),
    }

In [8]:
def scrape_audience_reviews(movie_name: str, max_reviews: int) -> List[Dict[str, Any]]:
    """Scrape up to `max_reviews` audience reviews for the given movie."""
    session = requests.Session()
    session.headers.update(HEADERS)

    print(f"Fetching movie page for '{movie_name}'...")
    html = fetch_movie_page(session, movie_name)
    movie_id = extract_movie_id(html)
    print(f"Found movie ID: {movie_id}")

    reviews: List[Dict[str, Any]] = []
    after_cursor = ""
    page_num = 0

    while len(reviews) < max_reviews:
        page_num += 1
        payload = fetch_reviews_page(session, movie_id, after_cursor=after_cursor, page_count=20)
        review_items = payload.get("reviews") or []
        page_info = payload.get("pageInfo") or {}

        if not review_items:
            print(f"No more reviews found on page {page_num}. Stopping.")
            break

        for item in review_items:
            normalized = normalize_review(item, movie_name)
            if normalized["review_text"]:
                reviews.append(normalized)
            if len(reviews) >= max_reviews:
                break

        print(f"Page {page_num}: collected {len(reviews)} reviews so far...")

        if len(reviews) >= max_reviews:
            break

        if not page_info.get("hasNextPage"):
            print("No more pages available.")
            break

        after_cursor = page_info.get("endCursor") or ""
        if not after_cursor:
            break

        time.sleep(0.5)  # be polite to the server

    return reviews[:max_reviews]

In [9]:
# ---- CONFIGURE THESE ----
MOVIE_NAME = "iron_man"   # use underscores instead of spaces
REVIEW_COUNT = 200         # number of audience reviews to scrape
# -------------------------

reviews = scrape_audience_reviews(MOVIE_NAME, REVIEW_COUNT)
print(f"\nTotal reviews scraped: {len(reviews)}")

Fetching movie page for 'iron_man'...
Found movie ID: 41ca006b-8820-379c-84fc-aaae870b37f6
Page 1: collected 10 reviews so far...
Page 2: collected 30 reviews so far...
Page 3: collected 50 reviews so far...
Page 4: collected 70 reviews so far...
Page 5: collected 90 reviews so far...
Page 6: collected 110 reviews so far...
Page 7: collected 130 reviews so far...
Page 8: collected 150 reviews so far...
Page 9: collected 170 reviews so far...
Page 10: collected 190 reviews so far...
Page 11: collected 200 reviews so far...

Total reviews scraped: 200


In [10]:
# Save to CSV
csv_filename = f"{MOVIE_NAME}_audience_reviews.csv"

if reviews:
    fieldnames = list(reviews[0].keys())
    with open(csv_filename, "w", newline="", encoding="utf-8-sig") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(reviews)
    print(f"Reviews saved to: {csv_filename}")
else:
    print("No reviews were collected — CSV not created.")

Reviews saved to: iron_man_audience_reviews.csv


In [11]:
movie = pd.read_csv(csv_filename)
ct = movie[['display_name', 'rating', 'review_text']]
ct

,display_name,rating,review_text
0,Jorge A,STAR_4_5,The movie that started the MCU. Still a great...
1,Weston A.,STAR_4,Phenomenal origin story and character developm...
2,Morgan L.,STAR_4,Pretty good story I felt like it was well pace...
3,John F,STAR_4_5,Amazing movie showing Tony starks arrogance in...
4,Guilherme M,STAR_5,RDJ perfomance as Tony Stark was impecable. Lo...
...,...,...,...
195,Kevin R,STAR_5,Phenomenal on so many levels and in so many ways.
196,NaN,STAR_5,This movie is absolutely spectacular. It’s sti...
197,Andy B,STAR_4,Rewatching this after many years the contrast ...
198,Cameron G,STAR_3,It’s pretty good I like iron man the story is ...


In [12]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(model_name)

sentiment_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

print("Model loaded successfully!")

Loading tokenizer...
Loading model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 27676.54it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!


In [13]:
# Quick test to make sure the model works
test_result = sentiment_pipeline("I love this movie!")
print(f"Test input:  'I love this movie!'")
print(f"Prediction:  {test_result[0]['label']} (confidence: {test_result[0]['score']:.4f})")

Test input:  'I love this movie!'
Prediction:  positive (confidence: 0.9847)


In [14]:
def tag(text):
    """Get the sentiment label for a given text."""
    # Truncate long reviews to 512 tokens (model max length)
    result = sentiment_pipeline(text[:512])
    return result[0]["label"]

print("Running sentiment analysis on all reviews... (this may take a moment)")
ct = ct.copy()                     # avoid SettingWithCopyWarning
ct["positive_or_negative"] = ct["review_text"].apply(tag)
print("Done!")


Running sentiment analysis on all reviews... (this may take a moment)


Done!


In [15]:
ct

,display_name,rating,review_text,positive_or_negative
0,Jorge A,STAR_4_5,The movie that started the MCU. Still a great...,positive
1,Weston A.,STAR_4,Phenomenal origin story and character developm...,positive
2,Morgan L.,STAR_4,Pretty good story I felt like it was well pace...,positive
3,John F,STAR_4_5,Amazing movie showing Tony starks arrogance in...,positive
4,Guilherme M,STAR_5,RDJ perfomance as Tony Stark was impecable. Lo...,positive
...,...,...,...,...
195,Kevin R,STAR_5,Phenomenal on so many levels and in so many ways.,positive
196,NaN,STAR_5,This movie is absolutely spectacular. It’s sti...,positive
197,Andy B,STAR_4,Rewatching this after many years the contrast ...,positive
198,Cameron G,STAR_3,It’s pretty good I like iron man the story is ...,positive


In [16]:
print("=" * 40)
print("Sentiment Distribution")
print("=" * 40)
print(ct['positive_or_negative'].value_counts())
print()
print(ct['positive_or_negative'].value_counts(normalize=True).apply(lambda x: f"{x:.1%}"))

Sentiment Distribution
positive_or_negative
positive    164
neutral      23
negative     13
Name: count, dtype: int64

positive_or_negative
positive    82.0%
neutral     11.5%
negative     6.5%
Name: proportion, dtype: str


In [17]:
output_with_sentiment = f"{MOVIE_NAME}_reviews_with_sentiment.csv"
ct.to_csv(output_with_sentiment, index=False, encoding="utf-8-sig")
print(f"Final results saved to: {output_with_sentiment}")

Final results saved to: iron_man_reviews_with_sentiment.csv
